# 03 — Evaluación
Evalúa los modelos **cargándolos desde `Models\`** (generados por `02_Modeling`), sin reentrenar. Estructura: métricas de regresión con validación cruzada, métricas de clustering para varios k, y evaluación del recomendador por Precisión@5 y cobertura.

In [ ]:
import joblib
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import (
    calinski_harabasz_score, davies_bouldin_score, silhouette_score,
)
from sklearn.model_selection import KFold, cross_val_predict, cross_val_score

estadisticas = pd.read_parquet("Data\\Clean\\estadisticas_carrera.parquet")
dataset_generica = pd.read_parquet("Data\\Clean\\dataset_carrera_generica.parquet")

art_ingresos = joblib.load("Models\\modelo_ingresos.joblib")
art_segmentacion = joblib.load("Models\\segmentacion.joblib")
config_reco = joblib.load("Models\\recomendador_config.joblib")

print("Modelo de ingresos:", art_ingresos["nombre_algoritmo"], "| n_train:", art_ingresos["n_train"])
print("Segmentación: k =", art_segmentacion["k"], "| n_train:", art_segmentacion["n_train"])

## 1. Evaluación — Predicción de ingresos

Se replica el esquema de validación cruzada (5 folds) sobre las 252 combinaciones para reportar **R²**, **MAE** (pesos) y **MAPE** (%), y se analizan los residuos por tipo de institución para detectar sesgos.

**Interpretación honesta:** con la carrera genérica como predictor, parte del ajuste proviene de memorizar el nivel de ingreso de cada carrera; el aporte real del modelo es interpolar combinaciones carrera × tipo de institución que SIES no publica. El MAE (~$250 mil sobre ingresos de $0,7—3,5 millones) dimensiona el error esperable de esas extrapolaciones.

In [ ]:
target = art_ingresos["target"]
features = art_ingresos["features"]

df_ing = estadisticas.copy()
for c in features:
    df_ing[c] = df_ing[c].astype(str)
df_ing[target] = df_ing[target].astype(float)
df_ing = df_ing.dropna(subset=[target])

X = art_ingresos["encoder"].transform(df_ing[features])
y = df_ing[target].values

cv = KFold(n_splits=5, shuffle=True, random_state=42)
modelo = art_ingresos["modelo"]

r2 = cross_val_score(modelo, X, y, cv=cv, scoring="r2").mean()
mae = -cross_val_score(modelo, X, y, cv=cv, scoring="neg_mean_absolute_error").mean()
y_pred_cv = cross_val_predict(modelo, X, y, cv=cv)
mape = float(np.mean(np.abs((y - y_pred_cv) / y)) * 100)

print(f"R2 (CV 5 folds):  {r2:.3f}")
print(f"MAE (CV 5 folds): ${mae:,.0f}")
print(f"MAPE (CV):        {mape:.1f}%")

In [ ]:
# Residuos por tipo de institución (sesgo sistemático del modelo)
residuos = pd.DataFrame({
    "Tipo de institución": df_ing["Tipo de institución"].values,
    "residuo": y - y_pred_cv,
})
residuos.groupby("Tipo de institución")["residuo"].agg(["mean", "std", "count"]).round(0)

## 2. Evaluación — Segmentación

**Silhouette** (separación de clusters, más alto mejor), **Davies-Bouldin** (más bajo mejor) y **Calinski-Harabasz** (más alto mejor) para k de 2 a 7, con el k elegido en `02` como referencia. Se caracteriza cada segmento para su lectura de negocio.

In [ ]:
feats_seg = art_segmentacion["features"]
df_seg = dataset_generica.dropna(subset=feats_seg).copy()
X_seg = art_segmentacion["scaler"].transform(df_seg[feats_seg])

tabla_clustering = []
for k in range(2, 8):
    labels = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(X_seg)
    tabla_clustering.append({
        "k": k,
        "Silhouette": round(silhouette_score(X_seg, labels), 4),
        "Davies-Bouldin": round(davies_bouldin_score(X_seg, labels), 4),
        "Calinski-Harabasz": round(calinski_harabasz_score(X_seg, labels), 1),
    })
tabla_clustering = pd.DataFrame(tabla_clustering)
print("k elegido en 02_Modeling:", art_segmentacion["k"])
tabla_clustering

In [ ]:
# Caracterización de los segmentos del modelo final (cargado desde Models\)
df_seg["segmento"] = art_segmentacion["kmeans"].predict(X_seg)
perfil = df_seg.groupby("segmento").agg(
    n_carreras=("Área Carrera Genérica", "count"),
    **{c: (c, "mean") for c in feats_seg},
).round(2)
perfil

## 3. Evaluación — Recomendador

Se define "relevante" como toda carrera con empleabilidad e ingreso sobre la mediana. Se simulan estudiantes con puntajes PAES de 450 a 850 en cada área de interés y se mide **Precisión@5** (proporción de recomendaciones relevantes) y **cobertura** (% del catálogo que aparece en al menos una recomendación).

In [ ]:
KEY = "Área Carrera Genérica"
COL_CORTE = config_reco["col_corte"]
pesos = config_reco["pesos"]

base = dataset_generica.dropna(subset=["Empleabilidad 1er año", "ingreso_4to_anio_valor"]).copy()
umbral_emp = base["Empleabilidad 1er año"].median()
umbral_ing = base["ingreso_4to_anio_valor"].median()
base["es_relevante"] = (
    (base["Empleabilidad 1er año"] >= umbral_emp) & (base["ingreso_4to_anio_valor"] >= umbral_ing)
)


def recomendar(df, puntaje, area=None, top_n=5):
    cand = df.copy()
    if area:
        cand = cand[cand["Área"] == area]
    con_corte = cand[COL_CORTE].notna()
    cand = cand[(con_corte & (cand[COL_CORTE] <= puntaje)) | ~con_corte]
    if cand.empty:
        return cand
    selectividad = (cand[COL_CORTE] / puntaje).fillna(0).clip(0, 1)
    cand["score"] = (
        cand["Empleabilidad 1er año"].rank(pct=True) * pesos["empleabilidad"]
        + cand["ingreso_4to_anio_valor"].rank(pct=True) * pesos["ingreso"]
        + selectividad.rank(pct=True) * pesos["selectividad"]
    )
    return cand.sort_values("score", ascending=False).head(top_n)


precisiones, recomendadas = [], set()
for area in base["Área"].dropna().unique():
    for puntaje in [450, 550, 650, 750, 850]:
        top5 = recomendar(base, puntaje, area)
        if len(top5) > 0:
            precisiones.append(top5["es_relevante"].mean())
            recomendadas.update(top5[KEY])

precision_5 = float(np.mean(precisiones))
cobertura = len(recomendadas) / len(base)
print(f"Escenarios simulados: {len(precisiones)} (áreas × puntajes)")
print(f"Precisión@5 promedio: {precision_5:.2%}")
print(f"Cobertura del catálogo: {cobertura:.2%} ({len(recomendadas)} de {len(base)} carreras)")

## 4. Tabla resumen — rendimiento de los 3 modelos

In [ ]:
resumen_final = pd.DataFrame([
    {
        "Modelo": "1. Recomendación de carreras",
        "Tipo": "Reglas de negocio + scoring",
        "Métrica principal": "Precisión@5",
        "Valor": round(precision_5, 3),
    },
    {
        "Modelo": f"2. Predicción de ingresos ({art_ingresos['nombre_algoritmo']})",
        "Tipo": "Regresión (área + carrera + tipo de institución)",
        "Métrica principal": "R² (CV 5 folds)",
        "Valor": round(float(r2), 3),
    },
    {
        "Modelo": f"3. Segmentación (k={art_segmentacion['k']})",
        "Tipo": "Clustering (K-Means)",
        "Métrica principal": "Silhouette Score",
        "Valor": art_segmentacion["silhouette"],
    },
])
resumen_final